# 🏯 百度贴吧爬虫 v2.0 - Colab 版

基于**移动端 API 优先**策略，双通道自动切换，反爬能力更强。

**使用方法：** 依次运行每个 Cell 即可。在 Cell 2 中修改你的爬取配置。

## 1️⃣ 安装依赖

In [ ]:
!pip install requests beautifulsoup4 lxml pandas -q
print('✅ 依赖安装完成')

## 2️⃣ 配置参数
修改下面的变量来控制爬取行为：

In [ ]:
# ──────── 基本配置 ────────
TIEBA_NAME = "三国志战略版"     # 🎯 贴吧名称
PREFER_MOBILE = True            # True=移动端优先(推荐)

# ──────── 爬取模式(三选一) ────────
# 模式1: 按页数
SCRAPE_MODE = "pages"
SCRAPE_PAGES = 3                # 爬取页数

# 模式2: 按小时 (取消下面两行注释，注释上面两行)
# SCRAPE_MODE = "hours"
# SCRAPE_HOURS = 24

# 模式3: 按天数
# SCRAPE_MODE = "days"
# SCRAPE_DAYS = 7

# ──────── 高级配置 ────────
INCLUDE_TOP = False             # 是否包含置顶帖
MAX_SCAN_PAGES = 20             # 时间模式最大扫描页数
DELAY_RANGE = (1.5, 3.5)        # 请求间隔(秒)
SAVE_TO_DRIVE = False           # True=保存到Google Drive
OUTPUT_FILENAME = "tieba_data.json"

print(f"✅ 配置完成: {TIEBA_NAME}吧 | 模式={SCRAPE_MODE}")

## 3️⃣ 加载爬虫核心代码
（直接运行，无需修改）

In [ ]:
import json
import random
import re
import time as _time
from datetime import datetime, timedelta
from typing import Optional
from urllib.parse import quote, urlencode

import requests
from bs4 import BeautifulSoup
from IPython.display import display, HTML as IPyHTML

# ---------- UA 池 ----------

MOBILE_UA_POOL = [
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like Mac OS X) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.6 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (Linux; Android 14; SM-S918B) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.6261.64 Mobile Safari/537.36",
    "Mozilla/5.0 (Linux; Android 13; Pixel 7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.6167.143 Mobile Safari/537.36",
    "Mozilla/5.0 (Linux; Android 14; SAMSUNG SM-A546B) "
    "AppleWebKit/537.36 (KHTML, like Gecko) SamsungBrowser/23.0 Chrome/115.0.0.0 Mobile Safari/537.36",
    "Mozilla/5.0 (Linux; Android 12; HarmonyOS; ALN-AL10) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.88 HuaWeiBrowser/13.0.5.303 Mobile Safari/537.36",
]

PC_UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
    "Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
]

POSTS_PER_PAGE_PC = 50
POSTS_PER_PAGE_MOBILE = 30


# ---------- 时间解析 ----------

def normalize_time(time_str: str) -> str:
    """统一贴吧各种时间格式"""
    now = datetime.now()
    s = time_str.strip()
    if not s:
        return ""

    if s.isdigit() and len(s) >= 9:
        try:
            return datetime.fromtimestamp(int(s)).strftime("%Y-%m-%d %H:%M:%S")
        except (ValueError, OSError):
            pass

    m = re.match(r"(\d+)\s*分钟前", s)
    if m:
        return (now - timedelta(minutes=int(m.group(1)))).strftime("%Y-%m-%d %H:%M")

    m = re.match(r"(\d+)\s*小时前", s)
    if m:
        return (now - timedelta(hours=int(m.group(1)))).strftime("%Y-%m-%d %H:%M")

    if "刚刚" in s:
        return now.strftime("%Y-%m-%d %H:%M")

    m = re.match(r"(\d{4})-(\d{1,2})-(\d{1,2})\s+(\d{1,2}):(\d{2})", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d} {int(m.group(4)):02d}:{m.group(5)}"

    m = re.match(r"^(\d{1,2}):(\d{2})$", s)
    if m:
        return f"{now.strftime('%Y-%m-%d')} {int(m.group(1)):02d}:{m.group(2)}"

    m = re.match(r"昨天\s*(\d{1,2}):(\d{2})", s)
    if m:
        y = now - timedelta(days=1)
        return f"{y.strftime('%Y-%m-%d')} {int(m.group(1)):02d}:{m.group(2)}"

    m = re.match(r"^(\d{1,2})-(\d{1,2})$", s)
    if m:
        month, day = int(m.group(1)), int(m.group(2))
        year = now.year
        if month > now.month or (month == now.month and day > now.day):
            year -= 1
        return f"{year}-{month:02d}-{day:02d}"

    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"

    m = re.match(r"^(\d{4})-(\d{1,2})$", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-01"

    return s


# ---------- 移动端解析器 ----------

class MobileParser:
    @staticmethod
    def parse(html: str) -> list[dict]:
        threads = []
        soup = BeautifulSoup(html, "lxml")

        # 策略A: 经典移动端HTML
        for item in soup.select("div.i, div.feed_item, div.tl_item"):
            t = MobileParser._parse_item(item)
            if t:
                threads.append(t)
        if threads:
            return threads

        # 策略B: SPA内嵌JSON
        for script in soup.find_all("script"):
            text = script.string or ""
            for pat in [
                r'window\.__INITIAL_DATA__\s*=\s*({.+?})\s*;',
                r'window\.__data\s*=\s*({.+?})\s*;',
                r'"thread_list"\s*:\s*(\[.+?\])\s*[,}]',
            ]:
                m = re.search(pat, text, re.DOTALL)
                if m:
                    try:
                        data = json.loads(m.group(1))
                        parsed = MobileParser._parse_json(data)
                        if parsed:
                            return parsed
                    except (json.JSONDecodeError, KeyError):
                        continue

        # 策略C: 降级到PC解析
        return PCParser.parse(html)

    @staticmethod
    def _parse_item(item) -> Optional[dict]:
        title_tag = item.select_one("a.ti, a.title, a[href*='/p/']")
        if not title_tag:
            return None
        title = title_tag.get_text(strip=True)
        if not title:
            return None

        href = title_tag.get("href", "")
        thread_id = 0
        m = re.search(r"/p/(\d+)", href)
        if m:
            thread_id = int(m.group(1))

        abs_tag = item.select_one(".abs, .feed_content, .ti_desc, .threadlist_abs")
        abstract = abs_tag.get_text(strip=True) if abs_tag else ""

        reply_count = 0
        rep_tag = item.select_one(
            ".rep_num, .reply_num, .feed_reply_num, span[class*='rep'], span[class*='reply']"
        )
        if rep_tag:
            num = re.sub(r"[^\d]", "", rep_tag.get_text())
            reply_count = int(num) if num else 0

        author_tag = item.select_one(".author, .feed_author, .un, a[href*='un='], span.name")
        author_name = author_tag.get_text(strip=True) if author_tag else ""

        time_tag = item.select_one(".time, .date, .feed_time, span[class*='time'], span[class*='date']")
        time_raw = time_tag.get_text(strip=True) if time_tag else ""

        is_top = bool(item.select_one(".icon_top, .top_tag, [class*='top']"))

        return {
            "thread_id": thread_id,
            "title": title,
            "abstract": abstract,
            "reply_count": reply_count,
            "is_top": is_top,
            "url": f"https://tieba.baidu.com/p/{thread_id}" if thread_id else "",
            "author": {"name": author_name, "name_id": author_name},
            "last_reply": {"author": "", "time": normalize_time(time_raw), "time_raw": time_raw},
            "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

    @staticmethod
    def _parse_json(data) -> list[dict]:
        thread_list = None
        if isinstance(data, list):
            thread_list = data
        elif isinstance(data, dict):
            for key in ("thread_list", "threads", "data"):
                if key in data:
                    val = data[key]
                    if isinstance(val, list):
                        thread_list = val
                        break
                    elif isinstance(val, dict) and "thread_list" in val:
                        thread_list = val["thread_list"]
                        break
        if not thread_list:
            return []

        threads = []
        for item in thread_list:
            if not isinstance(item, dict):
                continue
            tid = int(item.get("id", 0) or item.get("tid", 0) or 0)
            title = str(item.get("title", "") or item.get("thread_title", ""))
            if not title:
                continue

            abstract = item.get("abstract", "") or item.get("desc", "") or ""
            if isinstance(abstract, list):
                abstract = "".join(str(a.get("text", "")) for a in abstract if isinstance(a, dict))

            author_data = item.get("author", {}) or {}
            author_name = ""
            if isinstance(author_data, dict):
                author_name = str(author_data.get("name", "") or author_data.get("name_show", "") or "")

            last_time = str(
                item.get("last_time_int", "") or item.get("last_reply_time", "")
                or item.get("create_time", "") or ""
            )

            threads.append({
                "thread_id": tid,
                "title": title,
                "abstract": str(abstract),
                "reply_count": int(item.get("reply_num", 0) or 0),
                "is_top": bool(item.get("is_top", False)),
                "url": f"https://tieba.baidu.com/p/{tid}" if tid else "",
                "author": {"name": author_name, "name_id": author_name},
                "last_reply": {"author": "", "time": normalize_time(last_time), "time_raw": last_time},
                "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
        return threads


# ---------- PC端解析器 ----------

class PCParser:
    @staticmethod
    def parse(html: str) -> list[dict]:
        threads = []
        soup = BeautifulSoup(html, "lxml")
        li_list = soup.select("li.j_thread_list.clearfix")
        if not li_list:
            li_list = soup.select("#thread_list > li")
        for li in li_list:
            t = PCParser._parse_thread(li)
            if t:
                threads.append(t)
        return threads

    @staticmethod
    def _parse_thread(li) -> Optional[dict]:
        data = {}
        raw = li.get("data-field", "")
        if raw:
            try:
                data = json.loads(raw)
            except (json.JSONDecodeError, TypeError):
                pass

        title_tag = li.select_one("a.j_th_tit") or li.select_one(".threadlist_title a")
        if not title_tag:
            return None
        title = title_tag.get_text(strip=True)
        href = title_tag.get("href", "")
        if not title:
            return None

        thread_id = data.get("id", 0)
        if not thread_id and href:
            m = re.search(r"/p/(\d+)", href)
            if m:
                thread_id = int(m.group(1))

        abs_tag = li.select_one(".threadlist_abs, .threadlist_abs_onlyline")
        abstract = abs_tag.get_text(strip=True) if abs_tag else ""

        reply_count = data.get("reply_num", 0)
        if not reply_count:
            rep_tag = li.select_one(".threadlist_rep_num")
            if rep_tag:
                try:
                    reply_count = int(rep_tag.get_text(strip=True))
                except ValueError:
                    pass

        is_top = bool(data.get("is_top", False))
        if li.select_one(".icon_top, .threadlist_icon_top"):
            is_top = True

        author_name = data.get("author_name", "")
        if not author_name:
            a_tag = li.select_one(".frs-author-name-wrap a, .frs-author-name")
            if a_tag:
                author_name = a_tag.get_text(strip=True)

        last_author_tag = li.select_one(".j_replyer, span.j_replyer")
        last_author = last_author_tag.get_text(strip=True) if last_author_tag else ""

        time_tag = li.select_one(".threadlist_reply_date, .is_show_create_time, .j_reply_data")
        time_raw = time_tag.get_text(strip=True) if time_tag else ""
        if not time_raw:
            ts = data.get("last_time_int", "")
            if ts and str(ts).isdigit() and int(str(ts)) > 1000000000:
                time_raw = str(ts)

        return {
            "thread_id": thread_id,
            "title": title,
            "abstract": abstract,
            "reply_count": reply_count,
            "is_top": is_top,
            "url": f"https://tieba.baidu.com/p/{thread_id}" if thread_id else "",
            "author": {"name": author_name, "name_id": author_name},
            "last_reply": {"author": last_author, "time": normalize_time(time_raw), "time_raw": time_raw},
            "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }


# ---------- 核心爬虫 ----------

class TiebaScraper:
    """双通道贴吧爬虫 (Colab 版)"""

    MOBILE_URL = "https://tieba.baidu.com/mo/q/m"
    PC_URL = "https://tieba.baidu.com/f"

    def __init__(self, kw, delay=(1.5, 3.5), prefer_mobile=True):
        self.kw = kw
        self.delay = delay
        self._channel = "mobile" if prefer_mobile else "pc"
        self._fails = 0
        self.mobile_session = self._make_session(mobile=True)
        self.pc_session = self._make_session(mobile=False)

    @staticmethod
    def _make_session(mobile: bool) -> requests.Session:
        sess = requests.Session()
        ua = random.choice(MOBILE_UA_POOL if mobile else PC_UA_POOL)
        headers = {
            "User-Agent": ua,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "zh-CN,zh;q=0.9",
            "Accept-Encoding": "gzip, deflate, br",
            "Connection": "keep-alive",
        }
        if mobile:
            headers["Sec-CH-UA-Mobile"] = "?1"
            headers["Sec-CH-UA-Platform"] = '"Android"'
        else:
            headers["Sec-Fetch-Dest"] = "document"
            headers["Sec-Fetch-Mode"] = "navigate"
            headers["Upgrade-Insecure-Requests"] = "1"
        sess.headers.update(headers)
        return sess

    def _rotate_ua(self):
        self.mobile_session.headers["User-Agent"] = random.choice(MOBILE_UA_POOL)
        self.pc_session.headers["User-Agent"] = random.choice(PC_UA_POOL)

    def _build_url(self, page_num: int, channel: str) -> str:
        if channel == "mobile":
            params = {"kw": self.kw, "pn": page_num * POSTS_PER_PAGE_MOBILE, "lp": "5028"}
            return f"{self.MOBILE_URL}?{urlencode(params)}"
        else:
            params = {"kw": self.kw, "ie": "utf-8", "pn": page_num * POSTS_PER_PAGE_PC}
            return f"{self.PC_URL}?{urlencode(params)}"

    def _fetch(self, page_num: int, retry: int = 3) -> Optional[tuple]:
        channels = (
            ["mobile", "pc"] if self._channel == "mobile" else ["pc", "mobile"]
        )
        for ch in channels:
            sess = self.mobile_session if ch == "mobile" else self.pc_session
            for attempt in range(1, retry + 1):
                url = self._build_url(page_num, ch)
                try:
                    print(f"  📡 [{ch.upper():6s}] 第{page_num+1}页 (尝试{attempt}/{retry})")
                    resp = sess.get(url, timeout=15, allow_redirects=True)
                    resp.encoding = "utf-8"
                    if resp.status_code == 200 and len(resp.text) > 3000:
                        self._channel = ch
                        self._fails = 0
                        return resp.text, ch
                    print(f"  ⚠️  HTTP {resp.status_code}, 内容长度={len(resp.text)}")
                except requests.RequestException as e:
                    print(f"  ⚠️  异常: {e}")
                if attempt < retry:
                    _time.sleep(random.uniform(2, 5))
            print(f"  🔄 [{ch.upper()}] 通道失败，切换备用")
            self._rotate_ua()

        self._fails += 1
        print(f"  ❌ 第{page_num+1}页双通道均失败")
        return None

    def _parse(self, html: str, channel: str) -> list[dict]:
        if channel == "mobile":
            threads = MobileParser.parse(html)
        else:
            threads = PCParser.parse(html)
        return threads

    def scrape_by_pages(self, num_pages=1, include_top=False) -> list[dict]:
        all_threads, seen = [], set()
        for page in range(num_pages):
            result = self._fetch(page)
            if not result:
                if self._fails >= 3:
                    print("🛑 连续失败≥3次，终止爬取")
                    break
                continue
            html, ch = result
            threads = self._parse(html, ch)
            for t in threads:
                if t["thread_id"] in seen:
                    continue
                seen.add(t["thread_id"])
                if not include_top and t["is_top"]:
                    continue
                all_threads.append(t)
            print(f"  ✅ 第{page+1}/{num_pages}页 → 本页{len(threads)}条, 累计{len(all_threads)}条")
            if page < num_pages - 1:
                _time.sleep(random.uniform(*self.delay))
        return all_threads

    def scrape_by_time(self, hours=0, days=0, max_scan_pages=20, include_top=False) -> list[dict]:
        cutoff = datetime.now() - timedelta(hours=hours, days=days)
        print(f"  ⏰ 时间截止: {cutoff.strftime('%Y-%m-%d %H:%M')}")
        all_threads, seen = [], set()
        should_stop = False
        for page in range(max_scan_pages):
            result = self._fetch(page)
            if not result:
                if self._fails >= 3:
                    break
                continue
            html, ch = result
            added = 0
            for t in self._parse(html, ch):
                tid = t["thread_id"]
                if tid in seen:
                    continue
                seen.add(tid)
                if not include_top and t["is_top"]:
                    continue
                rt = t["last_reply"]["time"]
                if rt:
                    try:
                        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d"):
                            try:
                                dt = datetime.strptime(rt, fmt)
                                break
                            except ValueError:
                                continue
                        else:
                            all_threads.append(t); added += 1; continue
                        if dt < cutoff:
                            should_stop = True
                            continue
                    except Exception:
                        pass
                all_threads.append(t)
                added += 1
            print(f"  ✅ 第{page+1}页: +{added}条, 累计{len(all_threads)}条")
            if should_stop:
                print("  ⏹️ 已到达时间截止范围")
                break
            if page < max_scan_pages - 1:
                _time.sleep(random.uniform(*self.delay))
        return all_threads

print("✅ 爬虫核心代码加载完成")

## 4️⃣ 执行爬取

In [ ]:
print(f"🚀 开始爬取 [{TIEBA_NAME}吧] ...")
print(f"   模式: {SCRAPE_MODE} | 通道: {'移动端优先' if PREFER_MOBILE else 'PC端优先'}")
print("-" * 50)

scraper = TiebaScraper(
    kw=TIEBA_NAME,
    delay=DELAY_RANGE,
    prefer_mobile=PREFER_MOBILE,
)

if SCRAPE_MODE == "pages":
    threads = scraper.scrape_by_pages(num_pages=SCRAPE_PAGES, include_top=INCLUDE_TOP)
elif SCRAPE_MODE == "hours":
    threads = scraper.scrape_by_time(hours=SCRAPE_HOURS, max_scan_pages=MAX_SCAN_PAGES, include_top=INCLUDE_TOP)
elif SCRAPE_MODE == "days":
    threads = scraper.scrape_by_time(days=SCRAPE_DAYS, max_scan_pages=MAX_SCAN_PAGES, include_top=INCLUDE_TOP)
else:
    threads = []

print("-" * 50)
print(f"🎉 爬取完成！共获取 {len(threads)} 条帖子")

## 5️⃣ 查看结果

In [ ]:
import pandas as pd

if threads:
    flat = []
    for t in threads:
        flat.append({
            "标题": t["title"],
            "回复数": t["reply_count"],
            "作者": t["author"]["name"],
            "最后回复时间": t["last_reply"]["time"],
            "最后回复者": t["last_reply"]["author"],
            "摘要": (t["abstract"][:50] + "...") if len(t["abstract"]) > 50 else t["abstract"],
            "帖子链接": t["url"],
            "帖子ID": t["thread_id"],
        })
    df = pd.DataFrame(flat)
    print(f"📊 共 {len(df)} 条帖子:\n")
    display(df)
else:
    print("⚠️ 未获取到数据")

## 6️⃣ 保存数据

In [ ]:
from urllib.parse import quote

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_path = f"/content/drive/MyDrive/{OUTPUT_FILENAME}"
else:
    output_path = f"/content/{OUTPUT_FILENAME}"

result = {
    "meta": {
        "forum_name": f"{TIEBA_NAME}吧",
        "forum_url": f"https://tieba.baidu.com/f?kw={quote(TIEBA_NAME)}&ie=utf-8",
        "total_posts": len(threads),
        "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "scraper_version": "2.0.0-colab",
    },
    "posts": threads,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"✅ 已保存 {len(threads)} 条 → {output_path}")

if not SAVE_TO_DRIVE:
    try:
        from google.colab import files
        files.download(output_path)
        print("📥 下载已触发")
    except ImportError:
        print("(非 Colab 环境，跳过下载)")
    except Exception as e:
        print(f"下载触发失败: {e}，请从左侧文件面板手动下载")

## 7️⃣ 数据分析 (可选)

In [ ]:
if threads and len(threads) > 5:
    print(f"\n📈 数据概览 - {TIEBA_NAME}吧")
    print("=" * 50)
    print(f"  帖子总数:    {len(threads)}")
    print(f"  总回复数:    {df['回复数'].sum()}")
    print(f"  平均回复:    {df['回复数'].mean():.1f}")
    print(f"  最高回复:    {df['回复数'].max()}")
    print(f"  活跃作者数:  {df['作者'].nunique()}")

    print(f"\n🔥 回复数 TOP 5:")
    top5 = df.nlargest(5, "回复数")[["标题", "回复数", "作者"]].reset_index(drop=True)
    top5.index = top5.index + 1
    display(top5)

    print(f"\n👤 发帖最多的作者 TOP 5:")
    for name, count in df["作者"].value_counts().head(5).items():
        print(f"  {name}: {count}条")
else:
    print("数据量不足，跳过分析")